# Phase S1: Cooling Theory Validation (Enhanced)

## Goals
1. Validate BatchNorm as cooling mechanism
2. Measure activation variance to compute actual γ
3. Test heating effect (noise injection)
4. Controlled skip experiment (fixed depth)

## Theory
- γ = -k·ln(Var_out/Var_in) — cooling rate from variance reduction
- β(J, γ) = β₀(J) · φ(γ),  φ(γ) = e^(-γ/γ_c)/(1+γ/γ_c)
- Key: heating (negative γ) may NOT symmetrically reduce β

In [ ]:
# Setup
import os, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import linalg
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Clone if needed
if not os.path.exists('ThermoRG-NN'):
    !git clone https://github.com/xliu203/ThermoRG-NN.git
    %cd ThermoRG-NN
os.chdir('ThermoRG-NN')

In [ ]:
# Enhanced configs: norm × skip × (optional heating)
# NEW: Add heating configs (noise injection)
CONFIGS = [
    ('None_NoSkip',    'none',       False, 0.0),   # Baseline: no norm, no skip
    ('LN_NoSkip',     'layernorm',  False, 0.0),  # LayerNorm baseline
    ('BN_NoSkip',     'batchnorm',  False, 0.0),  # BatchNorm cooling
    ('Noise_NoSkip',   'none',       False, 0.1),   # NEW: Heating via noise (σ=0.1)
    ('NoiseHi_NoSkip', 'none',       False, 0.3),  # NEW: Strong heating (σ=0.3)
    ('None_Skip',     'none',       True,  0.0),   # Skip effect
    ('LN_Skip',       'layernorm',  True,  0.0),  # LN + Skip
    ('BN_Skip',       'batchnorm',  True,  0.0),   # BN + Skip
]

# D values via base channel width
D_VALUES = [32, 48, 64, 96]  # ~0.2M to ~2M params
SEEDS = [42, 123]
EPOCHS = 200
BATCH_SIZE = 128
LR = 0.01
WD = 5e-4
MOMENTUM = 0.9

OUT_DIR = Path('./phase_s1_results')
OUT_DIR.mkdir(exist_ok=True)
print(f"Configs: {len(CONFIGS)}, D values: {D_VALUES}")

## Network with Variance Tracking

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, noise_std=0.0, stride=1):
        super().__init__()
        self.norm_type = norm_type
        self.use_skip = use_skip
        self.noise_std = noise_std
        
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        
        if norm_type == 'layernorm':
            self.norm = nn.LayerNorm([out_ch, 32, 32])
        elif norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        else:
            self.norm = nn.Identity()
        
        self.act = nn.GELU()
        
        if use_skip and in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        elif use_skip:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = None
        
        # Variance tracking hooks
        self.var_in = 0.0
        self.var_out = 0.0
        self.count = 0

    def forward(self, x):
        # Track input variance
        self.var_in += x.detach().float().var().item()
        
        out = self.norm(self.conv(x))
        
        # Add heating noise if specified
        if self.noise_std > 0 and self.training:
            noise = torch.randn_like(out) * self.noise_std
            out = out + noise
        
        out = self.act(out)
        
        # Track output variance
        self.var_out += out.detach().float().var().item()
        self.count += 1
        
        if self.use_skip and self.skip is not None:
            out = out + self.skip(x)
        return out
    
    def get_variance_ratio(self):
        if self.count == 0:
            return 1.0
        return (self.var_out / self.count) / (self.var_in / self.count + 1e-8)
    
    def reset_variance(self):
        self.var_in = 0.0
        self.var_out = 0.0
        self.count = 0


class ValidationNet(nn.Module):
    def __init__(self, base_ch=64, norm_type='none', use_skip=False, noise_std=0.0, n_classes=10):
        super().__init__()
        channels = [3, base_ch, base_ch*2, base_ch*2]
        
        self.blocks = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.blocks.append(ConvBlock(
                channels[i], channels[i+1],
                norm_type=norm_type,
                use_skip=(i > 0 and use_skip),
                noise_std=noise_std,
                stride=1
            ))
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels[-1], n_classes)

    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]
    
    def get_variance_ratios(self):
        ratios = []
        for block in self.blocks:
            if isinstance(block, ConvBlock):
                ratios.append(block.get_variance_ratio())
        return ratios
    
    def reset_variance_tracking(self):
        for block in self.blocks:
            if isinstance(block, ConvBlock):
                block.reset_variance()
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

print("Network with variance tracking defined.")

## J_topo Computation

In [ ]:
def compute_D_eff(W):
    if W.dim() == 4:
        W = W.view(W.size(0), -1)
    fro_sq = (W ** 2).sum().item()
    S = linalg.svd(W.to('cpu')).S
    spec_sq = S[0].item() ** 2 + 1e-12
    return fro_sq / spec_sq

def compute_J_topo(weights, d_input=3.0):
    eta_vals = []
    d_prev = d_input
    for W in weights:
        if W.dim() == 4:
            W = W.view(W.size(0), -1)
        D_eff = compute_D_eff(W)
        eta = D_eff / max(d_prev, 1e-8)
        eta_vals.append(max(eta, 1e-8))
        d_prev = D_eff
    L = len(eta_vals)
    log_sum = sum(abs(math.log(e)) for e in eta_vals)
    return math.exp(-log_sum / L) if L > 0 else 0.0

# Compute gamma from variance ratios: γ = -k * ln(Var_ratio)
def compute_gamma(var_ratios, k=1.0):
    """γ = -k·ln(Var_out/Var_in)"""
    if not var_ratios:
        return 0.0
    gamma = 0.0
    for r in var_ratios:
        if r > 0:
            gamma += -k * math.log(r)
    return gamma / len(var_ratios)

print("J_topo and γ functions defined.")

## Data Loading

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10(root='./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10(root='./data', train=False, transform=transform_val, download=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## Training Function (with variance tracking)

In [ ]:
from scipy.optimize import curve_fit

def train_model(model, train_loader, val_loader, epochs, lr, wd, momentum, device, track_variance=True):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    
    # Reset variance tracking
    if track_variance:
        model.reset_variance_tracking()
    
    best_val_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)
        val_loss /= total
        val_acc = correct / total
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch + 1
    
    # Get variance ratios after training
    variance_ratios = []
    if track_variance:
        variance_ratios = model.get_variance_ratios()
    
    return {
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'variance_ratios': variance_ratios,
    }


def fit_scaling_law(losses_by_d, d_values):
    def power_law(D, alpha, beta, E):
        return alpha * np.array(D) ** (-beta) + E
    
    Ds = np.array(d_values)
    losses = np.array([losses_by_d[d] for d in d_values])
    
    try:
        popt, _ = curve_fit(power_law, Ds, losses, p0=[10.0, 0.5, 0.5],
                           bounds=([0, 0, 0], [1000, 5, 10]), maxfev=10000)
        alpha, beta, E = popt
        preds = power_law(Ds, alpha, beta, E)
        ss_res = ((losses - preds) ** 2).sum()
        ss_tot = ((losses - losses.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return {'alpha': float(alpha), 'beta': float(beta), 'E': float(E), 'R2': float(r2)}
    except Exception as e:
        return {'alpha': None, 'beta': None, 'E': None, 'R2': 0.0, 'error': str(e)}

print("Training functions defined.")

## Main Experiment Loop

In [ ]:
from datetime import datetime

all_results = {
    'timestamp': datetime.now().isoformat(),
    'config': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR, 'wd': WD,
               'd_values': D_VALUES, 'seeds': SEEDS},
    'configs': []
}

total_start = time.time()

for config_name, norm_type, use_skip, noise_std in CONFIGS:
    print(f"\n{'='*60}")
    print(f"[{config_name}] norm={norm_type}, skip={use_skip}, noise={noise_std}")
    print(f"{'='*60}")
    
    cfg_start = time.time()
    cfg_result = {
        'name': config_name,
        'norm': norm_type,
        'skip': use_skip,
        'noise_std': noise_std,
        'D_results': {}
    }

    # J_topo at init
    model_init = ValidationNet(base_ch=64, norm_type=norm_type, use_skip=use_skip, noise_std=noise_std)
    weights_init = model_init.get_conv_weights()
    J_topo_init = compute_J_topo(weights_init)
    cfg_result['J_topo_init'] = J_topo_init
    print(f"J_topo(init) = {J_topo_init:.4f}")
    del model_init; torch.cuda.empty_cache()

    # Train for each D
    for base_ch in D_VALUES:
        print(f"  base_ch={base_ch}...", end=' ', flush=True)
        d_result = {'base_ch': base_ch, 'seeds': {}}
        losses = []
        all_var_ratios = []

        for seed in SEEDS:
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            
            model = ValidationNet(base_ch=base_ch, norm_type=norm_type, use_skip=use_skip, noise_std=noise_std)
            result = train_model(model, train_loader, val_loader, EPOCHS, LR, WD, MOMENTUM, device)
            
            d_result['seeds'][seed] = {
                'best_val_loss': result['best_val_loss'],
                'best_val_acc': result['best_val_acc'],
                'best_epoch': result['best_epoch'],
                'variance_ratios': result['variance_ratios']
            }
            losses.append(result['best_val_loss'])
            all_var_ratios.extend(result['variance_ratios'])
            print('.', end='', flush=True)
            del model; torch.cuda.empty_cache()

        # Compute gamma from variance ratios
        avg_var_ratio = np.mean(all_var_ratios) if all_var_ratios else 1.0
        gamma = compute_gamma(all_var_ratios) if all_var_ratios else 0.0
        d_result['avg_var_ratio'] = avg_var_ratio
        d_result['gamma'] = gamma
        d_result['avg_val_loss'] = float(np.mean(losses))
        print(f" avg={d_result['avg_val_loss']:.4f}, γ={gamma:.4f}")
        cfg_result['D_results'][str(base_ch)] = d_result

    # Fit scaling law
    losses_by_d = {str(ch): cfg_result['D_results'][str(ch)]['avg_val_loss'] for ch in D_VALUES}
    fit = fit_scaling_law(losses_by_d, D_VALUES)
    cfg_result['scaling_fit'] = fit
    print(f"  Fit: α={fit.get('alpha','-'):.2f}, β={fit.get('beta','-'):.4f}, E={fit.get('E','-'):.4f}, R²={fit.get('R2','-'):.4f}")
    
    # Average gamma across all D values
    avg_gamma = np.mean([cfg_result['D_results'][str(ch)]['gamma'] for ch in D_VALUES])
    cfg_result['avg_gamma'] = avg_gamma
    print(f"  Average γ = {avg_gamma:.4f}")
    
    cfg_time = (time.time() - cfg_start) / 60
    cfg_result['wall_time_min'] = cfg_time
    print(f"  Time: {cfg_time:.1f} min")
    
    all_results['configs'].append(cfg_result)

total_time = (time.time() - total_start) / 60
all_results['total_time_min'] = total_time
print(f"\nTotal runtime: {total_time:.1f} min")

## Save Results

In [ ]:
# Save
out_file = OUT_DIR / 'phase_s1_results.json'
with open(out_file, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"Saved to {out_file}")

## Summary & Analysis

In [ ]:
# Summary table
print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)

baseline = next(c for c in all_results['configs'] if c['name'] == 'None_NoSkip')
base_beta = baseline['scaling_fit'].get('beta') or 0.4
base_gamma = baseline['avg_gamma'] or 0.0

print(f"{'Config':<15} {'J_topo':<8} {'γ':<8} {'β':<10} {'φ(β)':<10} {'E':<10}")
print("-" * 70)
for cfg in all_results['configs']:
    fit = cfg['scaling_fit']
    beta = fit.get('beta') or 0
    phi = beta / base_beta if base_beta > 0 else 0
    print(f"{cfg['name']:<15} {cfg['J_topo_init']:<8.4f} {cfg['avg_gamma']:<8.4f} {beta:<10.4f} {phi:<10.3f} {fit.get('E', '-'):<10.4f}")

In [ ]:
# Analysis
print("\n" + "="*70)
print("COOLING THEORY ANALYSIS")
print("="*70)

def get_cfg(name):
    return next((c for c in all_results['configs'] if c['name'] == name), None)

# 1. Normalization Cooling
print("\n1. Normalization Cooling:")
for name in ['None_NoSkip', 'LN_NoSkip', 'BN_NoSkip']:
    cfg = get_cfg(name)
    if cfg:
        beta = cfg['scaling_fit'].get('beta') or 0
        gamma = cfg['avg_gamma']
        phi = beta / base_beta if base_beta > 0 else 0
        print(f"   {name:<15}: β={beta:.4f}, γ={gamma:.4f}, φ={phi:.3f}")

# 2. Heating Effect (critical test of φ(-γ)=φ(γ))
print("\n2. Heating Effect (testing φ symmetry):")
none_cfg = get_cfg('None_NoSkip')
noise_cfg = get_cfg('Noise_NoSkip')
noisehi_cfg = get_cfg('NoiseHi_NoSkip')
if none_cfg and noise_cfg:
    none_beta = none_cfg['scaling_fit'].get('beta') or 0
    noise_beta = noise_cfg['scaling_fit'].get('beta') or 0
    noise_gamma = noise_cfg['avg_gamma']
    phi_noise = noise_beta / none_beta if none_beta > 0 else 0
    print(f"   None (γ≈0, heating=0): β={none_beta:.4f}")
    print(f"   Noise (γ≈{noise_gamma:.4f}, heating): β={noise_beta:.4f}, φ={phi_noise:.3f}")
    if noisehi_cfg:
        noisehi_beta = noisehi_cfg['scaling_fit'].get('beta') or 0
        noisehi_gamma = noisehi_cfg['avg_gamma']
        phi_noisehi = noisehi_beta / none_beta if none_beta > 0 else 0
        print(f"   NoiseHi (γ≈{noisehi_gamma:.4f}, strong heating): β={noisehi_beta:.4f}, φ={phi_noisehi:.3f}")
    print(f"   Prediction: if φ(-γ)=φ(γ), heating should REDUCE β")
    print(f"   But data may show: heating INCREASES β (asymmetric!)")

# 3. Skip Cooling
print("\n3. Skip Cooling (controlled):")
for name in ['None_Skip', 'LN_Skip', 'BN_Skip']:
    cfg = get_cfg(name)
    if cfg:
        beta = cfg['scaling_fit'].get('beta') or 0
        gamma = cfg['avg_gamma']
        phi = beta / base_beta if base_beta > 0 else 0
        print(f"   {name:<15}: β={beta:.4f}, γ={gamma:.4f}, φ={phi:.3f}")

# 4. γ vs β correlation
print("\n4. γ vs β Correlation:")
gammas = [c['avg_gamma'] for c in all_results['configs'] if c['scaling_fit'].get('beta')]
betas = [c['scaling_fit'].get('beta') for c in all_results['configs'] if c['scaling_fit'].get('beta')]
if len(gammas) > 2:
    corr = np.corrcoef(gammas, betas)[0, 1]
    print(f"   Pearson r(γ, β) = {corr:.4f}")
    print(f"   Prediction: γ ↑ → β ↓ (negative correlation expected)")

# 5. φ(γ) form test
print("\n5. Testing φ(γ) = e^(-γ/γ_c)/(1+γ/γ_c):")
print("   Need to fit γ_c to data...")